# Hazard discovery and fixed-SAM evaluation

Read saved artifacts on CPU. Development rows support tuning; final comparisons use test rows. Fixtures are software checks. Unscored nouns are not confirmed hallucinations. See `docs/hazard_evaluation.md` for metadata bindings and optional SAM/cost inputs.

In [ ]:
import json
import os
from pathlib import Path
import sys

REPO_ROOT_OVERRIDE = os.environ.get('TRAVERSABILITY_REPO_ROOT')
start = Path(REPO_ROOT_OVERRIDE or Path.cwd()).resolve()
candidates = [p for base in (start, *start.parents) for p in (base, base / 'VLM_evaluation')]
COMPONENT = next((p for p in candidates if (p / 'docs/hazard_interfaces.md').is_file()), None)
if COMPONENT is None:
    raise RuntimeError('Start within the checkout or set TRAVERSABILITY_REPO_ROOT.')
sys.path.insert(0, str(COMPONENT / 'src'))
from traversability_hazard_evaluation import evaluate, export_report

config = json.loads((COMPONENT / 'configs/hazards/evaluation.json').read_text(encoding='utf-8'))
config['component_root'] = str(COMPONENT)
# Roots can be set here or through TRAVERSABILITY_DATA_ROOT/RUN_ROOT.
config['run_name'] = 'hazard_prompt_v1'
config['fixture'] = False
# Enable only saved conditions/profiles; this notebook runs no models.
config['segmentation']['enabled'] = False
config['deployment']['profiles'] = []
EXPORT = False
OUTPUT_DIR = None  # Default: the evaluated run's evaluation/ directory.


In [ ]:
report = evaluate(config)
print(json.dumps({'fixture': report['fixture'], 'validation': report['validation'], 'coverage': report['coverage'], 'comparison': report['comparison']}, indent=2))


In [ ]:
# Inspect sources and splits separately; partial rows are diagnostic only.
for section in ('concepts', 'segmentation', 'deployment'):
    print(section, json.dumps(report.get(section, {}).get('rows', []), indent=2))
print('Missed concepts:', json.dumps(report.get('concepts', {}).get('missed_concepts', []), indent=2))


In [ ]:
if EXPORT:
    destination = Path(OUTPUT_DIR) if OUTPUT_DIR is not None else Path(report['identity']['run_dir']) / 'evaluation'
    paths = export_report(report, destination)
    print(json.dumps(paths, indent=2))
else:
    print('Export disabled; set EXPORT=True to save reports.')
